# Step 06c — LLM predictions → classified buildings

**Input:** `config.LLM_CHECKPOINT_FILE` (the production run of notebook 06b)
**Output:** `config.CLASSIFIED_BUILDINGS_FILE` with `CLASSIFIER_ARM='llm'`

Turns the LLM's flat prediction table into the exact GeoPackage that notebook 07
(redistribution) expects, so the LLM arm can reuse notebooks 07 and 08 unchanged.

**This is the production path only.** The benchmark does not come through here —
notebook 10 reads `06b_llm_predictions_*.parquet` directly, the same way it re-runs
`classify_building` in-process for the rule arm. Neither arm's score depends on a
GeoPackage.

## What this replaces

`windows` split this across three notebooks, two of which are broken:

| windows notebook | why it is not carried over |
|---|---|
| `07_llm_error_rerun` | builds `pois_map` keyed on int64 `gml_id` and looks it up with `str(gid)`, so **every retried row is sent an empty prompt**. The model dutifully answers "nothing here", `validate` passes, the row is checkpointed, and the notebook prints "All error rows resolved!". Re-running 06b retries the same rows under the identical prompt instead. |
| `08_attach_geometry` | re-attaches geometry but **drops `volume_m3`**, which its own redistribution notebook filters on — a `KeyError` in the pipeline as shipped. |
| `09_merge_results` | glob-merges multiple prediction files, which `append_parquet`'s `drop_duplicates(keep='last')` already does. |

## Contract with notebook 07

Matches `06_rule_based_classification` exactly: one row per building, `mid_label` as a
`str(list(...))` repr, `bosserhof_class` as a canonical string or **`None`**, plus
`volume_m3` and `geometry`.

`''` must become `None`: notebook 07 does `dropna(subset=['bosserhof_class'])`, which
does **not** drop empty strings — they survive to the weight lookup and produce a
spurious warning and a zero weight.

In [ ]:
import os
# MUST precede `from config import *` — `import *` binds the resolved paths into the
# kernel namespace, so a kernel that already imported config keeps the previous arm.
os.environ['CLASSIFIER_ARM'] = 'llm'

import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *
from llm_utils import normalise_mid_labels
from validation_utils import resolve_prediction_bosserhof

import pandas as pd
import geopandas as gpd

print(f'CLASSIFIER_ARM = {CLASSIFIER_ARM}')
print(f'  predictions -> {LLM_CHECKPOINT_FILE}')
print(f'  buildings   -> {CONDENSED_BUILDINGS_FILE.name}')
print(f'  writing     -> {CLASSIFIED_BUILDINGS_FILE.name}')

if not LLM_CHECKPOINT_FILE.exists():
    raise FileNotFoundError(
        f'{LLM_CHECKPOINT_FILE}\n  missing: the PRODUCTION run of notebook 06b.\n'
        '  The benchmark checkpoints (data/validation/06b_llm_predictions_*.parquet) are '
        'NOT interchangeable — they are keyed to the frozen file and describe different '
        'buildings under the same ids.')

---
## Step 1 — Load and normalise

Both normalisers are the ones the scorer uses, so a building classified here and the
same building scored in notebook 10 resolve identically.

In [ ]:
pred = pd.read_parquet(LLM_CHECKPOINT_FILE)
print(f'{len(pred):,} predictions')

pred['gml_id'] = pred['gml_id'].astype(str)
assert pred['gml_id'].is_unique, (
    'duplicate gml_id in the checkpoint — append_parquet dedups on write, so this '
    'means two runs over DIFFERENT building sets were merged into one file.')

if 'error' in pred.columns:
    n_err = int(pred['error'].notna().sum())
    assert n_err == 0, f'{n_err} error rows in the checkpoint; errors must never be checkpointed'

pred['mid_labels'] = pred['mid_labels'].map(normalise_mid_labels)
pred['bosserhof_class'] = pred['bosserhof_class'].map(resolve_prediction_bosserhof)

# '' means "explicitly no class" and must become None: notebook 07 drops NaN, not '',
# so an empty string survives into the weight lookup.
pred['bosserhof_class'] = pred['bosserhof_class'].where(
    pred['bosserhof_class'].notna() & (pred['bosserhof_class'] != ''), None)

n_cls = int(pred['bosserhof_class'].notna().sum())
print(f'with a Bosserhof class: {n_cls:,} ({n_cls / len(pred):.1%})')
print(f'with >=1 activity     : {int(pred["mid_labels"].map(len).gt(0).sum()):,}')

---
## Step 2 — Re-attach geometry and volume

`volume_m3` is carried explicitly. Dropping it is what makes `windows`' own pipeline
raise a `KeyError` at redistribution.

In [ ]:
buildings = gpd.read_file(CONDENSED_BUILDINGS_FILE)
buildings['gml_id'] = buildings['gml_id'].astype(str)
print(f'{len(buildings):,} buildings in {CONDENSED_BUILDINGS_FILE.name}')

out = buildings[['gml_id', 'volume_m3', 'geometry']].merge(
    pred[['gml_id', 'mid_labels', 'bosserhof_class', 'interpreted_type']],
    on='gml_id', how='inner', validate='one_to_one')

# Serialise exactly as notebook 06 does, so notebook 07 parses both arms identically.
# '[]' must survive as '[]' — it means "no activity here", not missing data.
out['mid_label'] = out['mid_labels'].map(lambda v: str(list(v)))
out = out.drop(columns=['mid_labels'])

out = gpd.GeoDataFrame(out, geometry='geometry', crs=buildings.crs)
CLASSIFIED_BUILDINGS_FILE.parent.mkdir(parents=True, exist_ok=True)
out.to_file(CLASSIFIED_BUILDINGS_FILE, driver='GPKG')
print(f'\nwrote {len(out):,} rows -> {CLASSIFIED_BUILDINGS_FILE.name}')

---
## Step 3 — Reconciliation

Notebooks 07 and 08 contain **no assertions at all** — a grep for `assert`/`raise`
across both returns nothing. Every schema mismatch there is a bare `KeyError` or a
silent row drop, so this is the last place a lost building is visible. Print the counts
rather than trusting the merge.

In [ ]:
n_pred, n_bld, n_out = len(pred), len(buildings), len(out)
print(f'predictions in        : {n_pred:,}')
print(f'buildings available   : {n_bld:,}')
print(f'joined (with geometry): {n_out:,}')
print(f'  predictions with no matching building: {n_pred - n_out:,}')
print(f'  buildings with no prediction         : {n_bld - n_out:,}')
print()
print(f'carrying a Bosserhof class : {int(out["bosserhof_class"].notna().sum()):,}')
print(f'carrying no activity ("[]"): {int((out["mid_label"] == "[]").sum()):,}')
print(f'geometry present           : {int(out.geometry.notna().sum()):,}')
print(f'volume_m3 present          : {int(out["volume_m3"].notna().sum()):,}')

if n_out < n_pred:
    print(f'\nWARNING: {n_pred - n_out:,} predictions dropped at the geometry join. If this '
          'is not zero, the checkpoint and the buildings file are from different runs — '
          'gml_id is a per-run positional index.')

print('\nNext: notebook 07 (redistribution) with CLASSIFIER_ARM=llm, then notebook 08.')